In [ ]:
import sys
from pathlib import Path

HICOLA_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'scripts' / 'hicola_paths.py').exists())
sys.path.insert(0, str(HICOLA_ROOT / 'scripts'))
import hicola_paths as hp

import numpy as np
import matplotlib.pyplot as plt
import h5py
import pynbody
import pynbody.analysis.halo as halo

In [ ]:
def get_xvm(s):
    x = s['pos'][:,0]
    y = s['pos'][:,1]
    z = s['pos'][:,2]
    vx = s['vel'][:,0]
    vy = s['vel'][:,1]
    vz = s['vel'][:,2]
    m = s['mass']
    return x,y,z,vx,vy,vz,m

In [ ]:
path = str(hp.standalone_run_dir('cuGal_test') / 'snapshot_cuGal_test_z0.000') + '/'
name = 'gadget_z0.000'
file = path + name
s = pynbody.load(file)
#s.physical_units()
#halo.center(s.dm)

In [ ]:
s['pos'].shape

In [ ]:
x,y,z,vx,vy,vz,m = get_xvm(s)
x,m

In [ ]:
def plot_2D_slice(s, zmin, zmax, physical=False):

    x,y,z,vx,vy,vz,m = get_xvm(s)
    
    Slice = ((zmin <= z) & (z < zmax))
    
    fig = plt.figure(figsize=(10, 8), dpi=300)
    ax = fig.add_subplot(111)
    
    p = ax.scatter(
        x[Slice], 
        y[Slice],
        c=np.log10(m[Slice]),
        cmap='viridis',
        s=10,
        alpha=0.8,
        edgecolors='k',
        linewidths=0.2
    )

    if physical:
        ax.set_xlabel('$x$ [kpc]', fontsize=14, labelpad=10)
        ax.set_ylabel('$y$ [kpc]', fontsize=14, labelpad=10)
    else:
        ax.set_xlabel('$x$ [Mpc/h]', fontsize=14, labelpad=10)
        ax.set_ylabel('$y$ [Mpc/h]', fontsize=14, labelpad=10)
        
    ax.tick_params(axis='both', which='major', labelsize=12)
    ax.tick_params(axis='both', which='minor', labelsize=10)
    
    cbar = fig.colorbar(p, ax=ax, pad=0.1)
    cbar.ax.tick_params(labelsize=10)

    if physical:
        cbar.set_label('$m$ [Msun]', fontsize=12)
    else:
        cbar.set_label('$m$ [Msun/h]', fontsize=12)
    
    ax.grid(True, linestyle='--', alpha=0.5)
    plt.tight_layout()
    plt.show()

In [ ]:
zmin = 50
zmax = 60
plot_2D_slice(s, zmin, zmax)

In [ ]:
def overdensity_field(s, Ngrid=256):
    """Compute mass density on a regular grid in a memory-safe way."""
    x,y,z,vx,vy,vz,m = get_xvm(s)
    npt = x.shape[0]
    # Domain extents
    xmin, xmax = x.min(), x.max()
    ymin, ymax = y.min(), y.max()
    zmin, zmax = z.min(), z.max()
    dx = (xmax - xmin) / float(Ngrid)
    dy = (ymax - ymin) / float(Ngrid)
    dz = (zmax - zmin) / float(Ngrid)

    # Allocate flattened histogram to keep memory small
    nbins = Ngrid * Ngrid * Ngrid
    hist_flat = np.zeros(nbins, dtype=np.float64)

    # Process particles in chunks to avoid large temporaries
    chunk = int(1e6)
    for start in range(0, npt, chunk):
        end = min(start + chunk, npt)
        xs = x[start:end]
        ys = y[start:end]
        zs = z[start:end]
        ws = m[start:end]

        ix = np.floor((xs - xmin) / dx).astype(np.int64)
        iy = np.floor((ys - ymin) / dy).astype(np.int64)
        iz = np.floor((zs - zmin) / dz).astype(np.int64)
        # Clip to valid range
        np.clip(ix, 0, Ngrid - 1, out=ix)
        np.clip(iy, 0, Ngrid - 1, out=iy)
        np.clip(iz, 0, Ngrid - 1, out=iz)

        flat_idx = ix * (Ngrid * Ngrid) + iy * Ngrid + iz
        binc = np.bincount(flat_idx, weights=ws, minlength=nbins)
        hist_flat += binc

    hist = hist_flat.reshape((Ngrid, Ngrid, Ngrid))
    edges = [
        np.linspace(xmin, xmax, Ngrid + 1),
        np.linspace(ymin, ymax, Ngrid + 1),
        np.linspace(zmin, zmax, Ngrid + 1),
    ]
    vcell = dx * dy * dz
    rho = hist / vcell
    return rho, edges

In [ ]:
Ngrid = 256
delta, edges = overdensity_field(s, Ngrid)
delta.shape

In [ ]:
def plot_2D_log_overdensity_field(s, Ngrid=256, physical=False):

    delta, edges = overdensity_field(s, Ngrid=Ngrid)
    delta_proj = np.mean(delta, axis=-1)
    #log_delta_proj = np.log10(1 + delta_proj)
    log_delta_proj = np.log10(delta_proj)

    fig = plt.figure(figsize=(10, 8), dpi=300)
    ax = fig.add_subplot(111)
    
    extent = [
    edges[0][0], edges[0][-1],
    edges[1][0], edges[1][-1]
    ]

    p = plt.imshow(log_delta_proj.T,
           origin='lower',
           extent=extent,
           #cmap='RdBu_r'
            cmap='magma'
                  )

    if physical:
        ax.set_xlabel('$x$ [kpc]', fontsize=14, labelpad=10)
        ax.set_ylabel('$y$ [kpc]', fontsize=14, labelpad=10)
    else:
        ax.set_xlabel('$x$ [Mpc/h]', fontsize=14, labelpad=10)
        ax.set_ylabel('$y$ [Mpc/h]', fontsize=14, labelpad=10)
    
    ax.tick_params(axis='both', which='major', labelsize=12)
    ax.tick_params(axis='both', which='minor', labelsize=10)
    
    cbar = fig.colorbar(p, ax=ax, pad=0.1)
    cbar.ax.tick_params(labelsize=10)
    if physical:
        cbar.set_label(r'$\log_{10}\delta_m$', fontsize=12)
    else:
        cbar.set_label(r'$\log_{10}\delta_m$', fontsize=12)
    
    plt.tight_layout()
    plt.savefig("fig/log_density.png")
    plt.show()

In [ ]:
plot_2D_log_overdensity_field(s, Ngrid=Ngrid)

In [ ]:
path_LCDM = str(hp.standalone_run_dir('LCDM_test') / 'snapshot_LCDM_test_z0.000') + '/'
name_LCDM = 'gadget_z0.000.0'
file_LCDM = path_LCDM + name_LCDM
s_LCDM = load_snap(file_LCDM)
#s_LCDM.physical_units()
#halo.center(s_LCDM.dm)

In [ ]:
zmin = 50
zmax = 150
plot_2D_slice(s_LCDM, zmin, zmax)

In [ ]:
Ngrid = 256
delta_LCDM, edges_LCDM = overdensity_field(s_LCDM, Ngrid)

In [ ]:
plot_2D_log_overdensity_field(s_LCDM, Ngrid=Ngrid)